# 📖 Notebook 3: Dashboard Design & Visualization

Metrics and alerts are useless if you can't **see** what's happening. Dashboards turn raw numbers into visual stories that let engineers spot problems in seconds.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to build dashboards in Grafana with Prometheus data
- Which panel types to use for different metrics (time-series, gauge, stat, table)
- How to write efficient PromQL queries for dashboards
- How query splitting and caching speed up dashboard loads
- Dashboard design best practices (the RED and USE methods)

## 🛠️ Setup

Make sure your infrastructure is running:

```bash
cd 06-system-designs/metrics-monitoring
docker compose up -d
```

**Important**: Run Notebook 1 first! The metrics server must be running so Prometheus and Grafana have data to display.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import requests
import time
import json
import math
import hashlib

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "metrics_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

PROMETHEUS_URL = "http://localhost:9090"
GRAFANA_URL = "http://localhost:3000"

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

def prom_query(query: str) -> list:
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={"query": query})
    data = resp.json()
    if data["status"] != "success":
        return []
    return data["data"]["result"]

def prom_range_query(query: str, start: float, end: float, step: str = "15s") -> list:
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query_range", params={
        "query": query, "start": start, "end": end, "step": step
    })
    data = resp.json()
    if data["status"] != "success":
        return []
    return data["data"]["result"]

# Test connections
try:
    resp = requests.get(f"{PROMETHEUS_URL}/-/healthy", timeout=5)
    print("✅ Connected to Prometheus")
except Exception as e:
    print(f"❌ Prometheus failed: {e}")

try:
    resp = requests.get(f"{GRAFANA_URL}/api/health", timeout=5)
    print("✅ Connected to Grafana")
except Exception as e:
    print(f"❌ Grafana failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")


## 🎨 Dashboard Design Methods

Before building panels, you need a **strategy** for what to show. Two popular methods:

### The USE Method (for infrastructure)

For every resource (CPU, memory, disk, network), monitor:

| Signal | What It Means | Example Metric |
|--------|--------------|----------------|
| **U**tilization | How busy is it? | `cpu_usage` at 85% |
| **S**aturation | How overloaded is it? | Queue depth at 500 |
| **E**rrors | Is it failing? | Disk I/O errors |

### The RED Method (for services)

For every service (API, database, queue), monitor:

| Signal | What It Means | Example Metric |
|--------|--------------|----------------|
| **R**ate | How many requests/sec? | `rate(http_requests_total[1m])` |
| **E**rrors | What % are failing? | `error_rate` at 3% |
| **D**uration | How long do they take? | `request_latency` p99 at 2s |

**A good dashboard uses BOTH**: USE for the boxes your services run on, RED for the services themselves.

## 🥇 The Four Golden Signals

USE and RED are great, but Google's SRE book proposes **Four Golden Signals** that combine the best of both and add one crucial signal most dashboards forget:

| Signal | What It Means | Example Query |
|--------|--------------|---------------|
| **Latency** | How long requests take (split success vs. failure!) | `histogram_quantile(0.99, sum by (le)(rate(http_duration_seconds_bucket[5m])))` |
| **Traffic** | How much demand you're getting | `sum(rate(http_requests_total[1m]))` |
| **Errors** | Rate of failed requests | `sum(rate(http_requests_total{status=~"5.."}[1m])) / sum(rate(http_requests_total[1m]))` |
| **Saturation** | How "full" your service is (queue depth, GC, CPU steal) | `max(queue_depth)` or `cpu_usage / cpu_limit` |

### Why Saturation Matters

CPU at 85% sounds fine. But if your request queue is at 9,990 out of 10,000 slots, you're one
burst away from dropping traffic. **Saturation leads latency** — it's how you catch problems
*before* they turn into errors your users can see.

### Two Traps in That Table

1. **Latency must come from a histogram, not a gauge.** Notebook 1 measured why: you cannot
   average percentiles across instances, and `histogram_quantile()` is only as precise as your
   bucket boundaries. `avg(latency)` on a dashboard is a comfort blanket — our own simulator's
   average latency sits around 0.45s while its p99 is nearly ten times that.
2. **Errors is a *ratio of two rates*, not an average of percentages.** `avg(error_rate)` across
   hosts weights a host serving 3 requests the same as one serving 30,000. Divide summed error
   rate by summed request rate instead, so busy hosts count for what they actually serve.

### Practical Recipe for a New Service

1. Start with exactly **four panels** on the overview — one per golden signal.
2. Each panel has a threshold line (your SLO).
3. If all four are green, the service is healthy. That's it. Everything else is a drill-down.


In [ ]:
# Let's build dashboard panels by querying Prometheus
# We'll simulate what Grafana does under the hood

print("📊 USE Method: Infrastructure Dashboard")
print("=" * 60)

# Utilization: CPU usage per host
print("\n🔹 Utilization (CPU)")
results = prom_query("demo_cpu_usage")
for r in results:
    host = r["metric"].get("host", "?")
    value = float(r["value"][1])
    bar = "█" * int(value / 5) + "░" * (20 - int(value / 5))
    print(f"  {host:<10} [{bar}] {value:.1f}%")

# Utilization: Memory usage
print("\n🔹 Utilization (Memory)")
results = prom_query("demo_memory_usage")
for r in results:
    host = r["metric"].get("host", "?")
    value = float(r["value"][1])
    bar = "█" * int(value / 5) + "░" * (20 - int(value / 5))
    print(f"  {host:<10} [{bar}] {value:.1f}%")

# Errors
print("\n🔹 Errors (Error Rate)")
results = prom_query("demo_error_rate")
for r in results:
    host = r["metric"].get("host", "?")
    value = float(r["value"][1])
    status = "🔴" if value > 5 else "🟡" if value > 1 else "🟢"
    print(f"  {status} {host:<10} {value:.2f}%")

In [ ]:
print("📊 RED Method: Service Dashboard")
print("=" * 60)

# Rate: requests per second
print("\n🔹 Rate (Requests/sec)")
results = prom_query('sum by (host)(rate(demo_http_requests_total[1m]))')
for r in results:
    host = r["metric"].get("host", "?")
    value = float(r["value"][1])
    print(f"  {host:<10} {value:>8.1f} req/s")

# Errors: fleet error rate as a RATIO OF RATES, not an average of per-host
# percentages. Averaging percentages gives a host serving 3 req/s the same vote
# as one serving 3,000 — the arithmetic is only correct when traffic is uniform,
# which it never is.
print("\n🔹 Errors (% of requests failing, request-weighted)")
err_ratio_q = ('100 * sum(rate(demo_http_requests_total{status="500"}[2m]))'
               ' / sum(rate(demo_http_requests_total[2m]))')
results = prom_query(err_ratio_q)
if results:
    fleet_err = float(results[0]["value"][1])
    status = "🔴" if fleet_err > 5 else "🟡" if fleet_err > 1 else "🟢"
    print(f"  {status} fleet      {fleet_err:.2f}%  (5xx rate ÷ total request rate)")
else:
    print("  ⏳ not enough scrapes yet for a 2-minute rate() window")

naive = prom_query('avg(demo_error_rate)')
if naive:
    print(f"  ℹ️  avg(demo_error_rate) says {float(naive[0]['value'][1]):.2f}% — a different")
    print("     number entirely, because it averages unweighted per-host percentages.")

# Duration: the average AND the p99, side by side. This is the single most
# important comparison on the page.
print("\n🔹 Duration (Request Latency)")
avg_lat = prom_query('avg(demo_request_latency)')
p99_q = ('histogram_quantile(0.99, '
         'sum by (le) (rate(demo_request_duration_seconds_bucket[2m])))')
p99_lat = prom_query(p99_q)

if avg_lat:
    v = float(avg_lat[0]["value"][1])
    print(f"  🟢 fleet average   {v:>7.3f}s   <- what avg(demo_request_latency) reports")
if p99_lat and p99_lat[0]["value"][1] not in ("NaN", "+Inf"):
    p99 = float(p99_lat[0]["value"][1])
    status = "🔴" if p99 > 2.0 else "🟡" if p99 > 0.5 else "🟢"
    print(f"  {status} fleet p99       {p99:>7.3f}s   <- what your slowest 1% of users live with")
    mean = float(avg_lat[0]["value"][1]) if avg_lat else 0.0
    if mean > 0:
        print(f"     The p99 is {p99 / mean:.1f}× the average. An alert")
        print("     on the average would never fire while those users time out.")
        print("     Note the p99 came from summing the `_bucket` series across hosts FIRST")
        print("     (`sum by (le)`), then taking the quantile. Never the other way round.")
else:
    print("  ⏳ histogram has no data yet — give Prometheus ~2 minutes of scrapes.")

print()
print("💡 The RED method tells you: Is the service handling requests?")
print("   Are they succeeding? Are they fast? That's all you need —")
print("   as long as 'fast' means a percentile and 'succeeding' means a weighted ratio.")


## 📈 Panel Types: Choosing the Right Visualization

| Panel Type | Best For | Example |
|-----------|---------|--------|
| **Time-series** | Trends over time | CPU usage over 24 hours |
| **Gauge** | Current value vs threshold | Current disk usage (75/100%) |
| **Stat** | Single important number | Total requests today |
| **Table** | Comparing many items | Top 10 slowest endpoints |
| **Heatmap** | Distribution over time | Latency percentile distribution |

### Rule of Thumb
- If you care about **change over time** → time-series
- If you care about **current status** → gauge or stat
- If you need to **compare items** → table

Let's build each type with real data!

In [ ]:
# Time-series panel: CPU usage over time
# This is what Grafana queries when rendering a time-series chart

end_time = time.time()
start_time = end_time - 300  # last 5 minutes

results = prom_range_query(
    'demo_cpu_usage{host="web-1"}',
    start=start_time,
    end=end_time,
    step="15s"
)

if results:
    values = results[0]["values"]
    print("📈 Time-Series Panel: CPU Usage (web-1, last 5 min)")
    print("=" * 60)
    print(f"   Data points: {len(values)}")
    print(f"   Step: 15 seconds")
    print()

    # ASCII sparkline
    nums = [float(v[1]) for v in values]
    min_v, max_v = min(nums), max(nums)
    width = 60
    bars = "▁▂▃▄▅▆▇█"

    # Show last 60 data points as ASCII chart
    display_values = nums[-width:]
    if max_v > min_v:
        sparkline = "".join(
            bars[min(len(bars) - 1, int((v - min_v) / (max_v - min_v) * (len(bars) - 1)))]
            for v in display_values
        )
    else:
        sparkline = bars[4] * len(display_values)

    print(f"   {max_v:>5.1f}% ┤")
    print(f"          │ {sparkline}")
    print(f"   {min_v:>5.1f}% ┤")
    print(f"          └{'─' * len(sparkline)}")
    print(f"           {'5 min ago':<30}{'now':>30}")
    print()
    print(f"   avg={sum(nums)/len(nums):.1f}%  min={min_v:.1f}%  max={max_v:.1f}%")
else:
    print("⏳ No data yet — wait for Prometheus to collect more samples.")

print()
print("👀 Open Grafana: http://localhost:3000")
print("   Check the pre-built 'Server Health Dashboard' for real charts!")

In [ ]:
# Gauge panel: current values with thresholds
# Shows whether something is in a good, warning, or critical range

print("📊 Gauge Panels: Current Status")
print("=" * 60)

gauges = [
    ("avg(demo_cpu_usage)", "Avg CPU", "%", [(0, 70, "🟢"), (70, 85, "🟡"), (85, 100, "🔴")]),
    ("avg(demo_memory_usage)", "Avg Memory", "%", [(0, 70, "🟢"), (70, 85, "🟡"), (85, 100, "🔴")]),
    ("avg(demo_error_rate)", "Error Rate", "%", [(0, 1, "🟢"), (1, 5, "🟡"), (5, 100, "🔴")]),
    ("avg(demo_request_latency)", "Avg Latency", "s", [(0, 0.5, "🟢"), (0.5, 2, "🟡"), (2, 100, "🔴")]),
]

for query, name, unit, thresholds in gauges:
    results = prom_query(query)
    if results:
        value = float(results[0]["value"][1])
        # Determine color based on thresholds
        icon = "❓"
        for low, high, i in thresholds:
            if low <= value < high:
                icon = i
                break
        print(f"  {icon} {name:<15} {value:>8.2f}{unit}")
    else:
        print(f"  ❓ {name:<15} no data")

print()
print("💡 Gauges with color thresholds let you spot problems at a glance.")
print("   Green = healthy, Yellow = investigate, Red = act now.")
print()
print("⚠️  Two of these four gauges are lying to you by construction:")
print("   'Avg Latency' hides the tail (compare it to the p99 above), and")
print("   'Error Rate' averages per-host percentages instead of weighting by")
print("   traffic. They are kept here because they are what most real dashboards")
print("   actually show — now you know why those dashboards stay green during")
print("   an incident.")


In [ ]:
# Table panel: compare all hosts side by side
# Great for finding outliers

print("📋 Table Panel: Host Comparison")
print("=" * 75)

# Collect all metrics for each host
host_data = {}

for metric, label in [("demo_cpu_usage", "cpu"), ("demo_memory_usage", "memory"),
                       ("demo_request_latency", "latency"), ("demo_error_rate", "errors")]:
    results = prom_query(metric)
    for r in results:
        host = r["metric"].get("host", "unknown")
        value = float(r["value"][1])
        if host not in host_data:
            host_data[host] = {}
        host_data[host][label] = value

print(f"{'Host':<10} {'CPU':>8} {'Memory':>8} {'Latency':>10} {'Errors':>8} {'Health'}")
print("-" * 75)

for host in sorted(host_data.keys()):
    d = host_data[host]
    cpu = d.get("cpu", 0)
    mem = d.get("memory", 0)
    lat = d.get("latency", 0)
    err = d.get("errors", 0)

    # Simple health score
    issues = sum([
        cpu > 85,
        mem > 85,
        lat > 2.0,
        err > 5.0
    ])
    health = ["🟢 Healthy", "🟡 Warning", "🟠 Degraded", "🔴 Critical", "💀 Down"][min(issues, 4)]

    print(f"{host:<10} {cpu:>7.1f}% {mem:>7.1f}% {lat:>9.3f}s {err:>7.2f}% {health}")

print()
print("💡 Tables are great for identifying which hosts are outliers.")
print("   Sort by any column to find the worst offenders.")

## ⚡ Query Optimization: Making Dashboards Fast

A dashboard with 10 panels, each querying 5 minutes of data at 15-second step, fires **10 queries** on every refresh. If the dashboard auto-refreshes every 5 seconds, that's **120 queries per minute**.

### Strategy 1: Query Splitting

Break a long-range query into:
- **Historical part** (e.g., 4h55m ago to 5m ago): cached, rarely changes
- **Recent part** (last 5m): always queried fresh

### Strategy 2: Result Caching

Cache query results in Redis with short TTLs (10-30 seconds).

### Strategy 3: Appropriate Step Size

Don't request 10-second data for a 30-day chart — use 1-hour steps!

In [ ]:
# Demonstrate query splitting: split a 5-minute query into cached + fresh

r = get_redis_client()


def split_windows(end: float, range_seconds: int, fresh_window: int, step: str) -> tuple:
    """
    Align a dashboard's time window DOWN onto the step grid.

    This alignment is the whole reason a query cache works. A key derived from a
    raw `time.time()` changes every second, so two panel refreshes one second
    apart would never share a cache entry — you'd get a 0% hit rate and never
    notice, because a cache that always misses still returns correct answers.
    Grafana and Mimir's query-frontend align to a fixed grid for exactly this
    reason.
    """
    step_seconds = int(step.rstrip("s"))
    aligned_end = math.floor(end / step_seconds) * step_seconds
    return aligned_end - range_seconds, aligned_end - fresh_window, aligned_end, step_seconds


def split_cache_key(query: str, start: float, split_point: float, step: str) -> str:
    """Cache key for the historical (immutable) half of a split query."""
    raw = f"{query}:{int(start)}:{int(split_point)}:{step}"
    return f"query_split:{hashlib.md5(raw.encode()).hexdigest()}"


def smart_range_query(query: str, end: float | None = None, range_seconds: int = 300,
                      fresh_window: int = 60, step: str = "15s",
                      ttl_seconds: int = 300) -> dict:
    """
    Smart query that splits into cached (old) + fresh (recent) parts.
    Only the fresh part hits Prometheus on every call.

    `end` is the dashboard's "to" timestamp. Grafana pins one `to` for the whole
    refresh and sends it to every panel; we do the same, so all panels in a
    refresh share cache entries instead of each minting its own.
    """
    if end is None:
        end = time.time()
    start, split_point, aligned_end, step_seconds = split_windows(
        end, range_seconds, fresh_window, step)

    stats = {"cache_hit": False, "historical_points": 0, "fresh_points": 0}

    # Part 1: Historical. Immutable for a pinned, aligned window — so it is safe
    # to cache it for far longer than the dashboard's refresh interval.
    cache_key = split_cache_key(query, start, split_point, step)
    cached = r.get(cache_key)

    if cached:
        historical = json.loads(cached)
        stats["cache_hit"] = True
    else:
        historical = prom_range_query(query, start, split_point, step)
        r.setex(cache_key, ttl_seconds, json.dumps(historical, default=str))

    if historical:
        stats["historical_points"] = len(historical[0].get("values", []))

    # Part 2: Fresh (always query Prometheus).
    # Start one step AFTER the split point: Prometheus range queries are inclusive
    # of both endpoints, so reusing split_point here would return a sample that the
    # historical half already contains and inflate total_points by exactly one.
    fresh = prom_range_query(query, split_point + step_seconds, aligned_end, step)
    if fresh:
        stats["fresh_points"] = len(fresh[0].get("values", []))

    stats["total_points"] = stats["historical_points"] + stats["fresh_points"]
    return stats


print("⚡ Query Splitting Demo")
print("=" * 60)

DEMO_Q = "avg(demo_cpu_usage)"
RANGE_S, FRESH_S, STEP = 300, 60, "15s"

# One dashboard refresh pins one `to` timestamp and hands it to every panel.
# Pinning it here also means the three calls below are guaranteed to address the
# same cache entry, instead of racing a second boundary.
dashboard_to = time.time()
_start, _split, _end, _ = split_windows(dashboard_to, RANGE_S, FRESH_S, STEP)
r.delete(split_cache_key(DEMO_Q, _start, _split, STEP))  # guarantee a cold start

timings, all_stats = [], []
for n in (1, 2, 3):
    t0 = time.time()
    s = smart_range_query(DEMO_Q, end=dashboard_to, range_seconds=RANGE_S,
                          fresh_window=FRESH_S, step=STEP)
    timings.append((time.time() - t0) * 1000)
    all_stats.append(s)
    print(f"Query {n}: historical={'🟢 HIT' if s['cache_hit'] else '🔴 MISS'} "
          f"total={s['total_points']} points  {timings[-1]:.0f}ms")

stats1, stats2, stats3 = all_stats
t1, t2, t3 = timings

if t2 < t1:
    print(f"\n📊 Query 2 was {t1 / t2:.1f}× faster — the historical half never left Redis.")
else:
    print(f"\n📊 Both calls were fast here ({t1:.0f}ms vs {t2:.0f}ms). Against a Prometheus")
    print("   holding months of data the cached half is what makes the panel usable;")
    print("   on a laptop with 5 minutes of data the round trip is already cheap.")
print()
print("💡 On query 2+, only the 'fresh' 60-second window hits Prometheus.")
print("   The historical 4 minutes are served from Redis cache.")
print("   Cache keys are built from the ALIGNED window, not from `now` — otherwise")
print("   every refresh would mint a fresh key and the hit rate would be zero.")

# The claim is about CACHING, not about milliseconds on a laptop, so that is what
# we assert. Because the window is pinned and aligned, this is exact, not a race.
hits = [s["cache_hit"] for s in all_stats]
assert hits == [False, True, True], (
    f"expected MISS, HIT, HIT against one pinned dashboard window — got {hits}. "
    f"A key derived from raw wall-clock time would look like this.")
if stats1["total_points"]:
    # A 300s range at a 15s step holds at most 300/15 + 1 = 21 points. The two
    # halves must partition it, not overlap: if the fresh half ever restarts AT
    # split_point instead of one step after it, this drifts to 22 and fails.
    max_points = (RANGE_S // 15) + 1
    assert stats1["total_points"] <= max_points, (
        f"split query returned {stats1['total_points']} points for a range that holds "
        f"at most {max_points} — the two halves are overlapping")


In [ ]:
# Demonstrate step size impact on query performance

print("📏 Step Size Impact on Data Volume")
print("=" * 60)

time_ranges = [
    ("Last 15 minutes", 900),
    ("Last 1 hour", 3600),
    ("Last 6 hours", 21600),
    ("Last 24 hours", 86400),
    ("Last 7 days", 604800),
    ("Last 30 days", 2592000),
]

step_sizes = [
    ("10s (raw)", 10),
    ("1m (rollup)", 60),
    ("1h (rollup)", 3600),
]

print(f"{'Time Range':<20}", end="")
for name, _ in step_sizes:
    print(f"{name:>15}", end="")
print()
print("-" * 65)

for range_name, seconds in time_ranges:
    print(f"{range_name:<20}", end="")
    for step_name, step in step_sizes:
        points = seconds // step
        if points > 100_000:
            display = f"{points/1000:.0f}K ⚠️"
        elif points > 10_000:
            display = f"{points/1000:.0f}K"
        else:
            display = f"{points:,}"
        print(f"{display:>15}", end="")
    print()

print()
print("💡 Smart step selection:")
print("   < 1 hour  → 10-15s step (raw data)")
print("   1-24 hours → 1 minute step")
print("   1-7 days   → 5 minute step")
print("   > 7 days   → 1 hour step")
print()
raw_30d = 2_592_000 // 10
hourly_30d = 2_592_000 // 3600
print(f"   A 30-day chart at 10s step = {raw_30d:,} points per series — unusable,")
print("   and pointless: your panel is maybe 1,000 pixels wide.")
print(f"   At 1h step = {hourly_30d} points — loads instantly, {raw_30d // hourly_30d}× less data.")

assert raw_30d == 259_200 and hourly_30d == 720, "step-size arithmetic drifted"


## 🏗️ Building a Dashboard in Grafana

We've pre-provisioned a dashboard in Grafana. Let's explore it!

### Pre-Built Dashboard

Open **http://localhost:3000** and look for the **"Server Health Dashboard"**.

It has 5 panels:
1. **CPU Usage by Host** — time-series showing all hosts
2. **Memory Usage by Host** — time-series showing memory trends
3. **Request Latency** — time-series for latency per host
4. **Error Rate** — time-series for errors
5. **HTTP Requests per Second** — rate of requests using `rate()` function

In [ ]:
# Let's programmatically create a new dashboard using Grafana's API
# This shows what happens "under the hood" when you build dashboards in the UI

dashboard_payload = {
    "dashboard": {
        # A stable uid makes `overwrite: True` actually overwrite. Without one,
        # Grafana treats every POST as a brand-new dashboard and you accumulate
        # a duplicate every time you re-run this cell.
        "uid": "red-method-demo",
        "title": "RED Method - Service Dashboard",
        "tags": ["demo", "red-method"],
        "timezone": "browser",
        "panels": [
            {
                "id": 1,
                "title": "Request Rate (per host)",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 12, "x": 0, "y": 0},
                "targets": [{
                    "expr": "sum by (host)(rate(demo_http_requests_total[1m]))",
                    "legendFormat": "{{ host }}"
                }],
                "fieldConfig": {
                    "defaults": {"unit": "reqps"},
                    "overrides": []
                }
            },
            {
                "id": 2,
                "title": "Error Rate (%)",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 12, "x": 12, "y": 0},
                "targets": [{
                    # Ratio of rates, not avg() of per-host percentages.
                    "expr": ('100 * sum(rate(demo_http_requests_total{status="500"}[2m]))'
                             ' / sum(rate(demo_http_requests_total[2m]))'),
                    "legendFormat": "fleet 5xx %"
                }],
                "fieldConfig": {
                    "defaults": {
                        "unit": "percent",
                        "thresholds": {
                            "steps": [
                                {"color": "green", "value": None},
                                {"color": "yellow", "value": 1},
                                {"color": "red", "value": 5}
                            ]
                        }
                    },
                    "overrides": []
                }
            },
            {
                "id": 3,
                "title": "Request Latency — p99 vs average",
                "type": "timeseries",
                "gridPos": {"h": 8, "w": 24, "x": 0, "y": 8},
                "targets": [
                    {
                        # Buckets summed across hosts FIRST, then the quantile.
                        "expr": ("histogram_quantile(0.99, sum by (le) "
                                 "(rate(demo_request_duration_seconds_bucket[2m])))"),
                        "legendFormat": "p99"
                    },
                    {
                        "expr": "avg(demo_request_latency)",
                        "legendFormat": "average (for contrast)"
                    }
                ],
                "fieldConfig": {
                    "defaults": {"unit": "s"},
                    "overrides": []
                }
            }
        ],
        "time": {"from": "now-15m", "to": "now"},
        "refresh": "10s"
    },
    "overwrite": True
}

try:
    resp = requests.post(
        f"{GRAFANA_URL}/api/dashboards/db",
        json=dashboard_payload,
        auth=("admin", "admin"),
        headers={"Content-Type": "application/json"}
    )
    if resp.status_code == 200:
        url = resp.json().get("url", "")
        print("✅ Created 'RED Method - Service Dashboard' in Grafana!")
        print(f"   Open: http://localhost:3000{url}")
    else:
        print(f"⚠️ Grafana API returned {resp.status_code}: {resp.text[:200]}")
except Exception as e:
    print(f"❌ Failed to create dashboard: {e}")

print()
print("💡 Grafana dashboards are just JSON! This means you can:")
print("   - Version control them in Git")
print("   - Auto-provision them via the API")
print("   - Share them as code, not screenshots")


## 🔍 Dashboard Design Best Practices

### The 5-Second Rule
A good dashboard should answer "is everything OK?" within 5 seconds of looking at it.

### Layout Tips

```
┌────────────────────────────────────────────┐
│  Row 1: BIG NUMBERS (stats/gauges)         │  ← Quick health check
│  [Total RPS] [Error Rate] [p99 Latency]    │
├────────────────────────────────────────────┤
│  Row 2: TIME-SERIES (trends)               │  ← What's changing?
│  [Request rate over time] [Errors over time]│
├────────────────────────────────────────────┤
│  Row 3: DETAILS (tables)                   │  ← Drill down
│  [Top errors by endpoint] [Slow queries]   │
└────────────────────────────────────────────┘
```

### Common Mistakes
1. **Too many panels** — 20+ panels = information overload
2. **No thresholds** — a line chart without context is meaningless
3. **Wrong time range** — 30-day view for debugging a live incident
4. **Missing labels** — "what does this line represent?"
5. **Not using rate()** — showing raw counter values instead of per-second rates

In [ ]:
# Let's look at the dashboard definitions stored in our database

conn = get_db_connection()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("SELECT name, description, panels FROM dashboards")
dashboards = cursor.fetchall()

for db in dashboards:
    panels = json.loads(db["panels"]) if isinstance(db["panels"], str) else db["panels"]
    print(f"📊 Dashboard: {db['name']}")
    print(f"   {db['description']}")
    print(f"   Panels: {len(panels)}")
    for i, panel in enumerate(panels, 1):
        print(f"     {i}. [{panel['type']}] {panel['title']}")
        print(f"        Query: {panel['query']}")

print()
print("💡 In production, dashboard configs are stored in Grafana's database.")
print("   But many teams also store them as JSON in Git for version control.")

conn.close()

## 🏛️ Putting It All Together: The Full Architecture

Here's how all the pieces we've covered connect:

```
┌────────────┐    ┌────────────┐    ┌────────────┐
│  Server 1   │    │  Server 2   │    │  Server N   │
│  (agent)    │    │  (agent)    │    │  (agent)    │
└──────┬──────┘    └──────┬──────┘    └──────┬──────┘
       │                  │                  │
       └──────────┬───────┴──────────────────┘
                  │  /metrics (HTTP)
                  ▼
         ┌────────────────┐
         │   Prometheus    │ ← Scrapes targets, stores time-series
         │   (TSDB)       │   Evaluates alert rules
         └────┬───────┬───┘
              │       │
    ┌─────────▼──┐  ┌─▼──────────────┐
    │   Grafana   │  │ Alert Manager  │
    │ (dashboards)│  │ (notifications)│
    └─────────────┘  └───────┬────────┘
                             │
                   ┌─────────┼─────────┐
                   ▼         ▼         ▼
               [Slack]  [PagerDuty] [Email]
```

### What We Covered

| Notebook | Topic | Key Concept |
|----------|-------|------------|
| 1 | Collection & Storage | Metrics → Prometheus → TSDB + Redis cache |
| 2 | Alerting | Rules → Evaluation loop → Notification service |
| 3 | Dashboards | Panels → PromQL → Query splitting + caching |

In [ ]:
# Final summary: verify everything is working

print("🏁 Lab Status Check")
print("=" * 50)

checks = [
    ("PostgreSQL", lambda: get_db_connection().close() or True),
    ("Redis", lambda: get_redis_client().ping()),
    ("Prometheus", lambda: requests.get(f"{PROMETHEUS_URL}/-/healthy").ok),
    ("Grafana", lambda: requests.get(f"{GRAFANA_URL}/api/health").ok),
    ("Prometheus has data", lambda: len(prom_query("demo_cpu_usage")) > 0),
]

all_ok = True
for name, check_fn in checks:
    try:
        if check_fn():
            print(f"  ✅ {name}")
        else:
            print(f"  ❌ {name}")
            all_ok = False
    except Exception as e:
        print(f"  ❌ {name}: {e}")
        all_ok = False

print()
if all_ok:
    print("🎉 Everything is running! You've built a complete metrics monitoring system.")
else:
    print("⚠️ Some services are down. Run: docker compose up -d")

print()
print("📍 Explore these URLs:")
print("   Grafana:        http://localhost:3000 (admin/admin)")
print("   Prometheus:     http://localhost:9090")
print("   Adminer:        http://localhost:8080")
print("   RedisInsight:   http://localhost:5540")
print()
print("🧹 When done, clean up with: docker compose down -v")

## 📚 Summary

### Key Takeaways

1. **USE method** (Utilization, Saturation, Errors) for infrastructure monitoring.
2. **RED method** (Rate, Errors, Duration) for service monitoring.
3. **Panel types** matter: time-series for trends, gauges for status, tables for comparison.
4. **Query splitting** caches historical data and only queries fresh data from the DB.
5. **Step size** must match the time range — don't request 10s data for a 30-day chart.
6. **Dashboard layout**: big numbers on top, time-series in the middle, details at the bottom.

### System Design Interview Tips

- **Always mention** the data flow: ingest → store → query → alert → notify
- **Cardinality** is the #1 scaling problem — too many unique label combinations
- **Rollups** solve long-range query performance — pre-compute 1m, 1h, 1d aggregates
- **Polling alerts** are simple and sufficient for most cases (Prometheus approach)
- **Separate write and read paths** — they have completely different performance needs
- **Meta-monitoring**: who monitors the monitoring system? (Use a separate simple watchdog)
- **Percentiles do not average** — merge histogram buckets across instances, *then* quantile

## 🌍 Real-World Examples

| Company | Stack | Interesting Choice |
|--------|------|--------------------|
| **Uber (M3)** | Custom TSDB on top of etcd + Cassandra | Built their own because Prometheus couldn't hold the cardinality of rider/driver/city labels |
| **Netflix (Atlas)** | In-memory TSDB, dimensional model | Keeps recent data in RAM for sub-second dashboard queries across millions of series |
| **Shopify / GitHub** | Prometheus + Thanos | Thanos bolts long-term object storage (S3) and global query onto vanilla Prometheus |
| **Datadog** | Push model, agent on every host | Agent aggregates locally to keep cardinality under control before shipping |
| **Grafana Cloud (Mimir)** | Horizontally-scaled Prometheus-compatible TSDB | Splits ingestion, storage, and query into separate services |

### What They All Agree On

1. **Separate write path from read path.** Writes are append-only and latency-insensitive; reads are ad-hoc and latency-critical.
2. **Downsample aggressively.** Raw data for days, not months.
3. **Cardinality is the enemy.** Every tier has guardrails that drop or reject high-cardinality labels.
4. **Alerts on symptoms, not causes.** Alert on "users can't check out," not "CPU > 80%."
5. **Alert on tail latency, not average latency.** Every one of these systems ships histograms
   for exactly this reason — an average is the one summary statistic guaranteed to look fine
   while your worst-served users leave.
